# STEP 01. 개발환경 확인
## 작업 계획
- Notebook이 .venv의 Python으로 실행되는지 확인한다.
- 필요한 패키지가 설치되어 있는지 확인한다.
## 이번에 하지 않는 것
- 크롤링 코드 작성

In [64]:
import sys
import platform

print("Python:", sys.version)
print("실행 위치:", sys.executable)   # .venv가 들어있어야 정상
print("Platform:", platform.platform())

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
실행 위치: c:\dev\claude-code-agent-course\chapter11\ax-job-agent\.venv\Scripts\python.exe
Platform: Windows-11-10.0.26200-SP0


In [65]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("BeautifulSoup import: OK")

pandas: 3.0.6
requests: 2.34.2
BeautifulSoup import: OK


# STEP 02. 수집 데이터 명세
## 작업 계획
- 수집할 채용공고 데이터의 컬럼(열)을 미리 정한다. (기준: `docs/SPEC.md` 6장)
- 컬럼 이름을 `COLUMNS` 리스트로 만들고, 9개가 맞는지 출력해서 확인한다.
## 이번에 하지 않는 것
- 크롤링, 인터넷 요청, 패키지 설치

**DataFrame의 한 행 = 채용공고 한 건**

| 컬럼 | 의미 | 예시 |
|---|---|---|
| `company_name` | 회사명 | ㈜에이아이랩 |
| `job_title` | 공고 제목 | 생성형 AI 엔지니어 채용 |
| `career` | 경력 조건 | 신입·경력 3년↑ |
| `location` | 근무 지역 | 서울 강남구 |
| `posted_date` | 등록일 | 2026-09-20 |
| `closing_date` | 마감일 | 2026-10-10 / 상시채용 |
| `job_url` | 공고 URL (**고유 키**) | https://www.jobkorea.co.kr/... |
| `search_keyword` | 어떤 검색어로 찾았는지 | LLM |
| `collected_at` | 수집 시각 | 2026-09-23 10:00 |

In [66]:
COLUMNS = [
    "company_name",    # 회사명
    "job_title",       # 공고 제목
    "career",          # 경력 조건
    "location",        # 근무 지역
    "posted_date",     # 등록일
    "closing_date",    # 마감일
    "job_url",         # 공고 URL (고유 키)
    "search_keyword",  # 어떤 검색어로 찾았는지
    "collected_at",    # 수집 시각
]

In [67]:
print("컬럼 개수:", len(COLUMNS))   # 9가 나와야 정상

for i, col in enumerate(COLUMNS, start=1):
    print(f"{i}. {col}")

컬럼 개수: 9
1. company_name
2. job_title
3. career
4. location
5. posted_date
6. closing_date
7. job_url
8. search_keyword
9. collected_at


## 실행 결과 해석
- 성공 여부: 성공
- 확인한 내용: 컬럼 9개가 SPEC.md 6장과 같은 순서로 출력됨 (company_name ~ collected_at)
- 예상과 다른 부분: 없음
- 다음 단계 진행 가능 여부: 가능

# STEP 03. 채용공고 페이지 접근 테스트
## 작업 계획
- 검색어 1개(`LLM`)로 잡코리아 검색 결과 페이지에 **딱 1번** 요청을 보낸다.
- 상태 코드, 응답 형식, 응답 길이를 보고 페이지에 접근할 수 있는지 확인한다.
- HTML 안에 검색어가 들어 있는지 확인한다. (데이터 추출은 다음 STEP에서 한다.)
## robots.txt 확인 결과
- `User-agent: *` 규칙에서 `/Search/` 경로는 막혀 있지 않다.
- AI 학습용 크롤러는 사이트 전체가 차단되어 있다. (이 프로젝트는 학습용 수집이 아니다.)
- 요청은 1번만 보낸다. (반복 요청이나 여러 페이지 요청은 하지 않는다.)
- 접근이 막히면 **우회하지 않고** 샘플 데이터로 전환한다.
## 이번에 하지 않는 것
- 반복 요청, 여러 페이지 요청, 상세 페이지 요청
- HTML에서 데이터 추출

In [68]:
KEYWORD = "LLM"   # 검색어 (처음에는 1개만 사용)

SEARCH_URL = "https://www.jobkorea.co.kr/Search/"   # 잡코리아 검색 페이지
params = {"stext": KEYWORD}   # 주소 뒤에 ?stext=LLM 으로 붙는 값

headers = {
    # 일반 브라우저(Chrome)처럼 보이는 User-Agent
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/128.0.0.0 Safari/537.36"
    ),
}

In [69]:
# 요청은 딱 1번만 보낸다 (이 셀을 여러 번 실행하지 않기)
response = requests.get(SEARCH_URL, params=params, headers=headers, timeout=10)

print("요청한 주소:", response.url)

요청한 주소: https://www.jobkorea.co.kr/Search/?stext=LLM


In [70]:
html = response.text

print("상태 코드:", response.status_code)   # 200이면 정상
print("Content-Type:", response.headers.get("Content-Type"))   # text/html 이면 정상
print("응답 길이(글자 수):", len(html))
print(f"HTML 안에 '{KEYWORD}' 등장 횟수:", html.count(KEYWORD))   # 0이면 검색 결과가 없거나 막힌 것일 수 있음
print("-" * 50)
print(html[:500])   # HTML 앞부분 500자

상태 코드: 200
Content-Type: text/html; charset=utf-8
응답 길이(글자 수): 341109
HTML 안에 'LLM' 등장 횟수: 191
--------------------------------------------------
<!DOCTYPE html><html lang="ko" data-sentry-component="RootLayout" data-sentry-source-file="layout.tsx"><head translate="no"><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1, user-scalable=no"/><link rel="preload" as="image" href="https://file2.jobkorea.co.kr/Net/Mng/Image/LogoImage?FN=2026/08/박정민캠페인최종본.png"/><link rel="stylesheet" href="https://frontend-app-cdn.jobkorea.co.kr/jobs/_next/static/css/49ef98337cc62b42.css" data-precedence="ne


## 실행 결과 해석
- 요청 성공 여부: 성공 (상태 코드 200, text/html)
- 확인한 데이터: 응답 길이 340,027자, HTML 안에 'LLM'이 191번 등장 → 검색 결과가 HTML에 들어 있음
- 예상과 다른 부분: 없음 (차단되지 않음)
- 다음 단계 진행 가능 여부: 가능 → STEP 04에서 공고 데이터 추출

# STEP 04. 소량 데이터 수집
## 04-1. HTML 저장
## 작업 계획
- STEP 03에서 받은 HTML(`response.text`)을 `data/raw/jobkorea_LLM_page1.html` 파일로 저장한다.
- 저장한 파일을 다시 읽어서 제대로 저장됐는지 확인한다.
## 이유
- 같은 페이지를 여러 번 요청하지 않으려고, 한 번 받은 HTML을 파일로 저장한다.
- 이후 STEP에서는 인터넷 요청 없이 **이 파일만** 사용한다.
## 이번에 하지 않는 것
- 새로운 인터넷 요청
- HTML에서 데이터 추출

In [71]:
from pathlib import Path

# Notebook은 notebooks/ 폴더에서 실행되므로, 한 칸 위가 프로젝트 루트
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_HTML_PATH = RAW_DIR / "jobkorea_LLM_page1.html"

if "response" not in globals():
    # 커널을 다시 시작해서 response가 없는 경우 (새로 요청하지 않음)
    print("STEP 03 요청 셀을 먼저 한 번 실행하세요")
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)   # 폴더가 없으면 만들고, 있으면 그대로
    RAW_HTML_PATH.write_text(response.text, encoding="utf-8")
    print("저장 완료:", RAW_HTML_PATH)

저장 완료: c:\dev\claude-code-agent-course\chapter11\ax-job-agent\data\raw\jobkorea_LLM_page1.html


In [72]:
saved_html = RAW_HTML_PATH.read_text(encoding="utf-8")   # 저장한 파일을 다시 읽기

print("파일 경로:", RAW_HTML_PATH)
print("파일 크기(KB):", round(RAW_HTML_PATH.stat().st_size / 1024, 1))
print("글자 수:", len(saved_html))   # STEP 03의 응답 길이와 같으면 정상

파일 경로: c:\dev\claude-code-agent-course\chapter11\ax-job-agent\data\raw\jobkorea_LLM_page1.html
파일 크기(KB): 362.9
글자 수: 341109


## 04-2. 공고 1건 추출
## 작업 계획
- 04-1에서 저장한 `data/raw/jobkorea_LLM_page1.html` 파일을 읽는다. (인터넷 요청 없음)
- 첫 번째 공고 1건에서 9개 컬럼을 꺼내 `first_job` dict로 만든다.
- 출력 결과를 브라우저의 잡코리아 화면과 비교한다.

## 어디서 무엇을 꺼내는가
| 컬럼 | 꺼내는 곳 | 방법 |
|---|---|---|
| `job_title` | 화면 카드 | 제목 링크(`Title`)의 글자 |
| `company_name` | 화면 카드 | 회사명 링크(카드의 마지막 공고 링크)의 첫 글자 |
| `location` | 화면 카드 | 회색 칩(`GrayChip`) 중 첫 번째 |
| `career` | 화면 카드 | 카드 글자 중 맨 마지막 |
| `job_url` | 화면 카드 | 공고 링크 `/Recruit/GI_Read/{공고번호}` (`?` 뒤는 버림) |
| `posted_date` | 숨은 JSON | 같은 공고번호의 `createdAt` 앞 10자리 (못 찾으면 None) |
| `closing_date` | 숨은 JSON | 같은 공고번호의 `applicationPeriod.end` 앞 10자리 (못 찾으면 None) |
| `search_keyword` | 설정값 | `KEYWORD` ("LLM") |
| `collected_at` | 파일 정보 | HTML 파일을 저장한 시각 |

## 이번에 하지 않는 것
- 인터넷 요청
- 2건 이상 추출 (04-3에서 한다)

In [73]:
import re
from datetime import datetime

html = RAW_HTML_PATH.read_text(encoding="utf-8")   # 인터넷 요청 없이 저장한 파일만 사용
soup = BeautifulSoup(html, "html.parser")

# 1) 화면 카드에서 추출
card = soup.find("div", attrs={"data-sentry-component": "CardJob"})   # 첫 번째 공고 카드
links = card.find_all("a", href=re.compile(r"/Recruit/GI_Read/\d+"))  # 로고·제목·회사명 링크 3개
title_link = card.find("a", attrs={"data-sentry-component": "Title"})
chips = card.find_all(attrs={"data-sentry-component": "GrayChip"})    # [지역, 직무]

job_url = links[0]["href"].split("?")[0]                        # ? 뒤(검색 기록용 값)는 버림
job_id = re.search(r"GI_Read/(\d+)", job_url).group(1)          # 공고번호
job_title = title_link.get_text(strip=True)
company_name = links[-1].find("span").get_text(strip=True)      # 회사명 링크의 첫 번째 글자
location = chips[0].get_text(" ", strip=True) if chips else None
career = list(card.stripped_strings)[-1]                        # 카드의 마지막 글자가 경력

# 2) 숨은 JSON에서 같은 공고번호로 날짜 찾기 (못 찾으면 None)
posted_date = None
closing_date = None
start = html.find(f'\\"id\\":\\"{job_id}\\"')
if start != -1:
    end = html.find('\\"legacyJobNo\\"', start + 50)             # 다음 공고가 시작되기 전까지만 보기
    job_json = html[start:end if end != -1 else start + 5000]
    m = re.search(r'createdAt\\":\\"(\d{4}-\d{2}-\d{2})', job_json)
    if m:
        posted_date = m.group(1)
    m = re.search(r'applicationPeriod\\":\{[^}]*end\\":\\"(\d{4}-\d{2}-\d{2})', job_json)
    if m:
        closing_date = m.group(1)

# 3) 9개 컬럼을 가진 dict 하나로 만들기
collected_at = datetime.fromtimestamp(RAW_HTML_PATH.stat().st_mtime).strftime("%Y-%m-%d %H:%M")   # 파일 저장 시각

first_job = {
    "company_name": company_name,
    "job_title": job_title,
    "career": career,
    "location": location,
    "posted_date": posted_date,
    "closing_date": closing_date,
    "job_url": job_url,
    "search_keyword": KEYWORD,
    "collected_at": collected_at,
}

In [74]:
for col in COLUMNS:   # STEP 02에서 정한 순서대로 출력
    print(f"{col}: {first_job[col]}")

company_name: 엔에이치엔㈜
job_title: [NHN] LLM 기술 개발 (LLM / Agent)
career: 경력2년↑
location: 경기 성남시
posted_date: 2026-09-07
closing_date: 2026-11-06
job_url: https://www.jobkorea.co.kr/Recruit/GI_Read/49941022
search_keyword: LLM
collected_at: 2026-09-23 12:44


## 04-2 실행 결과 해석
- 브라우저 화면과 비교 결과: 회사명·제목·경력·마감일 모두 일치
- 틀린 항목: 없음
- 다음 단계 진행 가능 여부: 가능 → 04-3에서 5~10건으로 확장

## 04-3. 공고 10건 추출
## 작업 계획
- 04-2의 추출 방법을 함수 `extract_job(card, html)`로 정리한다.
- 저장된 HTML 파일의 카드를 앞에서부터 읽어 공고 10건을 `rows` 리스트(dict 10개)에 담는다.
- 한 카드에서 오류가 나면 멈추지 않고, 그 카드는 건너뛰고 번호를 출력한다.

## 주의할 카드 모양 (04-2 방법에서 바꾼 점)
| 카드 모양 | 문제 | 처리 방법 |
|---|---|---|
| 복리후생 목록이 붙은 카드 | 경력이 맨 마지막 글자가 아님 | 제목 뒤의 글자 중 `경력`·`신입`·`무관`으로 시작하는 첫 글자를 경력으로 사용 |
| 회사명 옆에 그룹명이 붙은 카드 (예: 현대자동차그룹) | 회사명 링크 안에 글자가 2개 | 첫 번째 글자(회사명)만 사용 |
| 지역이 "서울 송파구 외 3"인 카드 | 지역이 여러 곳 | 그대로 둔다 |

- 제목 앞의 "신입 지원 가능" 배지가 경력으로 잘못 잡히지 않도록, 경력은 **제목 뒤에서만** 찾는다.

## 이번에 하지 않는 것
- 인터넷 요청
- 다음 페이지(21번째 공고부터) 추출

In [75]:
import re
from datetime import datetime

MAX_JOBS = 10   # 이번에 추출할 공고 수

html = RAW_HTML_PATH.read_text(encoding="utf-8")   # 인터넷 요청 없이 저장한 파일만 사용
soup = BeautifulSoup(html, "html.parser")
collected_at = datetime.fromtimestamp(RAW_HTML_PATH.stat().st_mtime).strftime("%Y-%m-%d %H:%M")   # 파일 저장 시각


def extract_job(card, html):
    """공고 카드 1개에서 9개 컬럼을 꺼내 dict로 돌려준다."""
    # 1) 화면 카드에서 추출
    links = card.find_all("a", href=re.compile(r"/Recruit/GI_Read/\d+"))  # 로고·제목·회사명 링크 3개
    title_link = card.find("a", attrs={"data-sentry-component": "Title"})
    chips = card.find_all(attrs={"data-sentry-component": "GrayChip"})    # [지역, 직무]

    job_url = links[0]["href"].split("?")[0]                    # ? 뒤(검색 기록용 값)는 버림
    job_id = re.search(r"GI_Read/(\d+)", job_url).group(1)      # 공고번호
    job_title = title_link.get_text(strip=True)
    company_name = links[-1].find("span").get_text(strip=True)  # 첫 번째 글자 = 회사명 (두 번째는 그룹명)
    location = chips[0].get_text(" ", strip=True) if chips else None   # "서울 송파구 외 3"도 그대로

    # 경력: 제목 뒤의 글자 중 "경력"/"신입"/"무관"으로 시작하는 첫 글자
    # (복리후생 목록이 붙은 카드는 경력이 맨 마지막 글자가 아니라서)
    strings = list(card.stripped_strings)
    after_title = strings[strings.index(job_title) + 1:]       # 제목 앞의 배지("신입 지원 가능")는 제외
    career = next((s for s in after_title if re.fullmatch(r"(경력|신입|무관)\S*", s)), None)

    # 2) 숨은 JSON에서 같은 공고번호로 날짜 찾기 (못 찾으면 None)
    posted_date = None
    closing_date = None
    start = html.find(f'\\"id\\":\\"{job_id}\\"')
    if start != -1:
        end = html.find('\\"legacyJobNo\\"', start + 50)         # 다음 공고가 시작되기 전까지만 보기
        job_json = html[start:end if end != -1 else start + 5000]
        m = re.search(r'createdAt\\":\\"(\d{4}-\d{2}-\d{2})', job_json)
        if m:
            posted_date = m.group(1)
        m = re.search(r'applicationPeriod\\":\{[^}]*end\\":\\"(\d{4}-\d{2}-\d{2})', job_json)
        if m:
            closing_date = m.group(1)

    return {
        "company_name": company_name,
        "job_title": job_title,
        "career": career,
        "location": location,
        "posted_date": posted_date,
        "closing_date": closing_date,
        "job_url": job_url,
        "search_keyword": KEYWORD,
        "collected_at": collected_at,
    }


cards = soup.find_all("div", attrs={"data-sentry-component": "CardJob"})   # 페이지의 공고 카드 전체
rows = []      # 추출한 공고(dict)를 담을 리스트
skipped = []   # 오류로 건너뛴 카드 번호

for i, card in enumerate(cards, start=1):
    if len(rows) >= MAX_JOBS:   # 10건이 모이면 멈춤
        break
    try:
        rows.append(extract_job(card, html))
    except Exception as e:
        skipped.append(i)
        print(f"{i}번 카드 건너뜀: {e}")   # 멈추지 않고 다음 카드로

In [76]:
print("추출 건수:", len(rows))                        # 10이 나와야 정상
print("건너뛴 건수:", len(skipped), skipped)          # 0이 나와야 정상 (건너뛴 카드 번호)
print("-" * 50)

for i, row in enumerate(rows, start=1):
    print(f"{i}. {row['company_name']} | {row['job_title']} | {row['career']} | {row['location']} | {row['closing_date']}")

추출 건수: 10
건너뛴 건수: 0 []
--------------------------------------------------
1. 엔에이치엔㈜ | [NHN] LLM 기술 개발 (LLM / Agent) | 경력2년↑ | 경기 성남시 | 2026-11-06
2. 현대오토에버㈜ | LLM 엔지니어 | 경력5년↑ | 서울 강남구 | 2026-09-28
3. ㈜쿡앱스 | [쿡앱스] LLM & 벡엔드 엔지니어 | 경력무관 | 경기 성남시 | 2026-10-23
4. ㈜카카오 | LLM Research Engineer (Post-training) (신입/경력) | 신입·경력 | 경기 성남시 | 2026-10-30
5. ㈜카카오 | AI Research Engineer (Search & Agent) (경력) - Agentic Search LLM | 경력 | 경기 성남시 | 2026-10-26
6. 주식회사 솔트룩스 | [솔트룩스]AI/LLM 엔지니어 경력직 채용(5년 이상) | 경력5년↑ | 서울 송파구 외 3 | 2070-01-01
7. ㈜씨어스 | [경력] LLM Developer | 경력5년↑ | 경기 성남시 | 2026-10-16
8. ㈜노리스페이스(NoriSpace Co.,Ltd) | LLM Research Engineer 모집 | 신입·경력 | 서울 영등포구 | 2026-10-10
9. 뷰스컴퍼니 | [바이브 코딩] LLM 사내 자동화 개발자 | 경력 | 서울 강남구 | 2026-10-07
10. ㈜솔트룩스 | [AI서비스사업본부] AI/LLM 백엔드 개발자 경력직 채용 | 경력5년↑ | 서울 송파구 외 2 | 2070-01-01


## 04-3 실행 결과 해석
- 추출 건수: 10건 (건너뜀 0건)
- 브라우저와 비교한 공고 번호: 4번(카카오), 6번(솔트룩스)
- 틀린 항목: 없음. 단, 6·10번 마감일 2070-01-01 = 화면상 "상시채용" → STEP 06에서 처리
- 다음 단계 진행 가능 여부: 가능 → STEP 05 DataFrame 생성

In [77]:
print(len(rows))

10


# STEP 05. DataFrame 생성
## 작업 계획
- 04-3에서 만든 `rows`(공고 dict 10개)를 pandas DataFrame `df`로 바꾼다.
- 컬럼 순서는 STEP 02의 `COLUMNS`를 그대로 사용한다.
- 행·열 개수, 앞 5행, 컬럼별 빈 값과 자료형을 확인한다.

> **DataFrame이란?** 파이썬 안의 **엑셀 표**. 한 행 = 채용공고 한 건, 한 열 = 컬럼(회사명, 제목 …) 하나.

## 이번에 하지 않는 것
- 인터넷 요청
- 데이터 정제 (날짜 변환, 중복 제거, 2070-01-01 처리 등은 STEP 06에서 한다)

In [78]:
import pandas as pd   # 표(DataFrame)를 다루는 도구

df = pd.DataFrame(rows, columns=COLUMNS)   # rows(dict 10개)를 표로 바꾸기, 열 순서는 STEP 02의 COLUMNS

In [79]:
print(df.shape)   # (행 수, 열 수) → (10, 9)이면 정상

display(df.head())   # 앞 5행

df.info()   # 컬럼별 빈 값이 아닌 개수(Non-Null Count)와 자료형(Dtype)

print("컬럼 순서가 COLUMNS와 같은가:", list(df.columns) == COLUMNS)   # True이면 정상

(10, 9)


,company_name,job_title,career,location,posted_date,closing_date,job_url,search_keyword,collected_at
0,엔에이치엔㈜,[NHN] LLM 기술 개발 (LLM / Agent),경력2년↑,경기 성남시,2026-09-07,2026-11-06,https://www.jobkorea.co.kr/Recruit/GI_Read/499...,LLM,2026-09-23 12:44
1,현대오토에버㈜,LLM 엔지니어,경력5년↑,서울 강남구,2026-09-02,2026-09-28,https://www.jobkorea.co.kr/Recruit/GI_Read/499...,LLM,2026-09-23 12:44
2,㈜쿡앱스,[쿡앱스] LLM & 벡엔드 엔지니어,경력무관,경기 성남시,2026-08-24,2026-10-23,https://www.jobkorea.co.kr/Recruit/GI_Read/498...,LLM,2026-09-23 12:44
3,㈜카카오,LLM Research Engineer (Post-training) (신입/경력),신입·경력,경기 성남시,2026-08-31,2026-10-30,https://www.jobkorea.co.kr/Recruit/GI_Read/498...,LLM,2026-09-23 12:44
4,㈜카카오,AI Research Engineer (Search & Agent) (경력) - A...,경력,경기 성남시,2026-08-27,2026-10-26,https://www.jobkorea.co.kr/Recruit/GI_Read/498...,LLM,2026-09-23 12:44


<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   company_name    10 non-null     str  
 1   job_title       10 non-null     str  
 2   career          10 non-null     str  
 3   location        10 non-null     str  
 4   posted_date     10 non-null     str  
 5   closing_date    10 non-null     str  
 6   job_url         10 non-null     str  
 7   search_keyword  10 non-null     str  
 8   collected_at    10 non-null     str  
dtypes: str(9)
memory usage: 852.0 bytes
컬럼 순서가 COLUMNS와 같은가: True


## STEP 05 실행 결과 해석
- df.shape 결과: (10, 9) → 공고 10건, 컬럼 9개
- 빈 값이 있는 컬럼: 없음 (모든 컬럼 10 non-null)
- 자료형에서 눈에 띄는 점: 모든 컬럼이 str(글자). 날짜도 글자라서 STEP 06에서 날짜형으로 변환 필요
- 다음 단계 진행 가능 여부: 가능

In [80]:
print(df.shape)

(10, 9)


# STEP 06. 전처리 / 중복 제거
## 작업 계획
- 원본 `df`는 그대로 두고, 복사본 `clean_df`를 만들어 작업한다.
- 먼저 빈 값과 중복을 **확인만** 하고, 그다음 `clean_df`에서 정리한다.
- 정리 전후를 비교해서 잘못 바뀐 곳이 없는지 확인한다.

## 할 일 4가지
| 할 일 | 무엇을 | 어떻게 |
|---|---|---|
| 빈 값 확인 | 컬럼별로 비어 있는 칸 수 | `df.isna().sum()` (확인만, 채우거나 지우지 않음) |
| 중복 제거 | 같은 `job_url`이 여러 번 나온 공고 | `job_url` 기준으로 앞의 것만 남김 |
| 날짜 변환 | `posted_date`, `closing_date`, `collected_at` (지금은 글자) | `pd.to_datetime(..., errors="coerce")`로 날짜형으로 (변환 못 하면 빈 값) |
| 상시채용 처리 | 마감일이 `2070-01-01`인 공고 | 새 컬럼 `is_always_open = True`, 마감일은 빈 값으로 |

## 이번에 하지 않는 것
- 인터넷 요청, 원본 `df` 수정
- 파일 저장 (CSV 저장은 STEP 07에서 한다)

In [81]:
# 확인만 하고 바꾸지 않음
print("컬럼별 빈 값 개수")
print(df.isna().sum())   # 0이면 빈 칸 없음
print("-" * 50)
print("job_url 기준 중복 개수:", df.duplicated(subset="job_url").sum())   # 앞에 같은 URL이 이미 있는 행의 수

컬럼별 빈 값 개수
company_name      0
job_title         0
career            0
location          0
posted_date       0
closing_date      0
job_url           0
search_keyword    0
collected_at      0
dtype: int64
--------------------------------------------------
job_url 기준 중복 개수: 0


In [82]:
ALWAYS_OPEN_DATE = "2070-01-01"                             # 잡코리아가 상시채용에 넣는 마감일
DATE_COLUMNS = ["posted_date", "closing_date", "collected_at"]   # 날짜형으로 바꿀 컬럼

clean_df = df.copy()                                         # 원본 df는 그대로 두고 복사본으로 작업

clean_df = clean_df.drop_duplicates(subset="job_url", keep="first")   # 같은 job_url은 앞의 것만 남김

clean_df["is_always_open"] = clean_df["closing_date"] == ALWAYS_OPEN_DATE   # 상시채용이면 True, 아니면 False
clean_df.loc[clean_df["is_always_open"], "closing_date"] = None             # 상시채용의 마감일은 빈 값으로

for col in DATE_COLUMNS:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")   # 글자 → 날짜형 (변환 못 하면 빈 값 NaT)

In [83]:
print("행 수: 원본 df", len(df), "→ clean_df", len(clean_df))   # 차이 = 제거된 중복 수
print("-" * 50)

print(clean_df.dtypes)   # 날짜 3개 컬럼이 datetime64 이면 정상
print("-" * 50)

always_open = clean_df[clean_df["is_always_open"]]
print("상시채용 건수:", len(always_open))
for _, row in always_open.iterrows():
    print(" -", row["company_name"], "|", row["job_title"])
print("-" * 50)

# 날짜 변환 실패: 원래 값이 있었는데 변환 후 빈 값이 된 수 (상시채용은 일부러 비웠으니 제외)
original = df.loc[clean_df.index]   # clean_df에 남은 행과 같은 행의 원본 값
for col in DATE_COLUMNS:
    had_value = original[col].notna()
    if col == "closing_date":
        had_value = had_value & ~clean_df["is_always_open"]
    failed = (had_value & clean_df[col].isna()).sum()
    print(f"{col} 변환 실패:", failed)   # 0이면 정상

display(clean_df[["company_name", "posted_date", "closing_date", "is_always_open"]])

행 수: 원본 df 10 → clean_df 10
--------------------------------------------------
company_name                 str
job_title                    str
career                       str
location                     str
posted_date       datetime64[us]
closing_date      datetime64[us]
job_url                      str
search_keyword               str
collected_at      datetime64[us]
is_always_open              bool
dtype: object
--------------------------------------------------
상시채용 건수: 2
 - 주식회사 솔트룩스 | [솔트룩스]AI/LLM 엔지니어 경력직 채용(5년 이상)
 - ㈜솔트룩스 | [AI서비스사업본부] AI/LLM 백엔드 개발자 경력직 채용
--------------------------------------------------
posted_date 변환 실패: 0
closing_date 변환 실패: 0
collected_at 변환 실패: 0


,company_name,posted_date,closing_date,is_always_open
0,엔에이치엔㈜,2026-09-07,2026-11-06,False
1,현대오토에버㈜,2026-09-02,2026-09-28,False
2,㈜쿡앱스,2026-08-24,2026-10-23,False
3,㈜카카오,2026-08-31,2026-10-30,False
4,㈜카카오,2026-08-27,2026-10-26,False
5,주식회사 솔트룩스,2026-09-21,NaT,True
6,㈜씨어스,2026-09-16,2026-10-16,False
7,"㈜노리스페이스(NoriSpace Co.,Ltd)",2026-08-11,2026-10-10,False
8,뷰스컴퍼니,2026-09-07,2026-10-07,False
9,㈜솔트룩스,2026-08-20,NaT,True


## 전처리 규칙 정리 (나중에 `src/preprocess.py`로 옮길 때 참고)
| 순서 | 규칙 | 코드 |
|---|---|---|
| 0 | 원본은 건드리지 않고 복사본으로 작업 | `clean_df = df.copy()` |
| 1 | `job_url`이 같으면 같은 공고 → 앞의 것만 남김 | `drop_duplicates(subset="job_url", keep="first")` |
| 2 | 마감일 `2070-01-01` = 상시채용 → `is_always_open = True` | `closing_date == "2070-01-01"` |
| 3 | 상시채용의 마감일은 빈 값으로 (가짜 날짜가 통계에 섞이지 않게) | `loc[is_always_open, "closing_date"] = None` |
| 4 | 날짜 3개 컬럼은 날짜형으로, 변환 못 하면 빈 값(NaT) | `pd.to_datetime(..., errors="coerce")` |
| 5 | 빈 값은 확인만 하고 채우거나 지우지 않음 | `isna().sum()` |

- 순서가 중요: **2 → 3을 4보다 먼저** 해야 한다. (날짜형으로 바꾼 뒤에는 글자 `"2070-01-01"`과 비교되지 않음)
- 변환 실패 건수는 "원래 값이 있었는데 빈 값이 된 수"로 센다. (상시채용은 일부러 비운 것이라 제외)

## STEP 06 실행 결과 해석
- 빈 값: 없음 (모든 컬럼 0)
- 중복: 0건 (행 수 10 → 10)
- 날짜 변환 실패: 0건 (posted_date, closing_date, collected_at 모두 datetime으로 변환)
- 상시채용 건수: 2건 (솔트룩스). 마감일은 NaT로 비우고 is_always_open=True로 표시
- 다음 단계 진행 가능 여부: 가능